# CSFeatures V1.0 用户示例
本示例演示如何读取预处理后的 AnnData 数据、识别细胞类型特异性特征并保存结果。输入对象的 `obs` 中应包含细胞类型标签；批次分析时还应包含批次字段。

In [ ]:
from pathlib import Path
import scanpy as sc
import marker_utils

adata = sc.read_h5ad('./example_data/rna.h5ad')
assert 'celltype' in adata.obs.columns
adata

In [ ]:
# 不含批次字段时，直接计算每个细胞类型的 EI 排名。
info, result_adata = marker_utils.getMarkersEI(
    adata=adata,
    celltype_key='celltype',
    n_comps=50,
    n_neighbors=30,
    random_state=0,
)

In [ ]:
# 查看指定细胞类型的前 10 个候选特征。
target_celltype = 'B'
top10 = (
    info[target_celltype]
    .sort_values('EI', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10

In [ ]:
# 保存完整排名表和结果 AnnData。
output_dir = Path('./results')
marker_dir = output_dir / 'markers'
marker_dir.mkdir(parents=True, exist_ok=True)

for celltype, table in info.items():
    table.sort_values('EI', ascending=False).to_csv(
        marker_dir / f'{celltype}_markers.csv', index=False
    )
result_adata.write_h5ad(output_dir / 'csfeatures_result.h5ad', compression='gzip')

## 可选：按批次生成中位秩共识列表
当 `obs` 中包含批次字段时，CSFeatures 在各批次的未整合归一化矩阵上独立计算 EI 排名，再按特征的跨批次中位秩生成共识结果。

In [ ]:
consensus, per_batch, batch_adatas = marker_utils.getMarkersEI(
    adata=adata,
    celltype_key='celltype',
    batch_col='batch',
    random_state=0,
)

consensus['B'].head(10)